# L9b: Pricing American Style Options Contracts using the Cox, Ross, Rubinstein (CRR) Binomial Model
Previously, we priced European options, which can be exercised only at expiration. However, standard options on stocks and ETFs are American-style and can be exercised at any time. What is the right to exercise early worth?

> __Learning Objectives:__
> 
> By the end of this lecture, you should be able to:
> * __Distinguish between European and American options:__ Understand the key differences in exercise timing and pricing implications between European options (exercisable only at expiration) and American options (exercisable at any time before or on expiration).
> * __Apply the binomial model to price American options:__ Use the CRR model to construct binomial trees and compute option premiums, incorporating the decision rule for early exercise.
> * __Estimate the premium of American options by simulation:__ Use the Longstaff–Schwartz method to estimate the same premium by Monte Carlo simulation, regressing the value of holding on the share price.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L9b) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples

__We will work through these examples in lecture:__

> [▶ Let's compute the Premium for Market-Quoted American Call and Put Options using CRR](CHEME-5660-L9b-Example-AmericanContractsPremium-Fall-2026.ipynb). In this example, we'll compute the price of American call and put option contracts using the CRR binomial model, compare our results to observed market prices, and measure each contract's early-exercise premium.

> [▶ Let's compute the premium of an American put by Monte Carlo](CHEME-5660-L9b-Example-LSM-Premium-Fall-2026.ipynb). In this example, we'll work the Longstaff–Schwartz regression by hand on eight paths, then price an American put by simulation and compare it with the CRR lattice and the European premium.

__Review this example on your own.__

> [▶ Let's compute the Premium for Hull's Hypothetical American Put using a binomial lattice](CHEME-5660-L9b-Example-Hull-American-Put-Fall-2026.ipynb). In this example, we'll compute the price of a hypothetical American put option contract using backward induction over a binomial lattice model, and compare our results to the same contract written as a European contract.
___

## Company Profile: The Options Clearing Corporation

[The Options Clearing Corporation (OCC)](https://www.theocc.com/) stands between the buyer and the seller of every U.S. listed option. It opened on April 26, 1973, as CBOE Clearing Corporation, a subsidiary of the new Chicago Board Options Exchange. In 1975, it became OCC, an [industry clearinghouse for all U.S. equity options](https://annualreport.theocc.com/2023/celebrating-50-years), headquartered in Chicago.

> __What does the firm do?__
>
> * __Exercise and assignment:__ When a holder exercises an American option early, the exercise notice reaches OCC through a clearing member, a firm that settles trades directly with OCC. OCC [assigns each notice at random](https://www.optionseducation.org/referencelibrary/faq/options-assignment) to a clearing member with a short position in the same contract, and that member passes it to one of its customers.
> * __Automatic exercise at expiration:__ OCC introduced exercise by exception in 1977, the year it began clearing puts. Under OCC Rule 805, an expiring equity option that is [in the money by 0.01 USD/share or more](https://www.optionseducation.org/referencelibrary/faq/options-exercise) is exercised unless the holder's clearing member instructs otherwise.
> * __Monte Carlo margin and binomial trees:__ OCC sets the collateral (margin) its clearing members post with [STANS](https://www.theocc.com/risk-management/margin-methodology), a large-scale Monte Carlo system in use since August 2006. STANS simulates 10,000 scenarios of share prices and implied volatilities over two days and reprices every contract in each one. American options are [repriced on a binomial tree](https://www.govinfo.gov/content/pkg/FR-2019-07-09/pdf/2019-14627.pdf), because the Black–Scholes–Merton formula cannot value early exercise.

__Explore further:__

* __Jobs and internships:__ Browse OCC's [careers page](https://www.theocc.com/careers) and its [internship program](https://www.theocc.com/careers/gain-experience-with-occ) in Chicago and Dallas/Fort Worth. The summer 2027 postings include an internship in Quantitative Risk Management, the group that builds OCC's margin models.
* __Videos:__ The [Options Industry Council](https://www.optionseducation.org/), which OCC formed with the exchanges in 1992, publishes a [YouTube channel](https://www.youtube.com/@OptionsEdu) on listed options. Start with [Option Exercise and Assignment](https://www.youtube.com/embed/OC-gkRp9hlw) and [The Automatic Exercise Process Explained](https://www.youtube.com/embed/cFckMpLdXPE).

__Connection to today's lecture:__ A seller of an American put can be assigned once a holder finds that exercising now beats holding. A margin based on the European premium would leave out the early-exercise premium, which is largest for puts deep in the money.
___

## American Style Options Contracts
American-style options contracts allow the holder to exercise the option __at any time__ before or on the expiration date. Previously, we wrote the premium of an American-style contract as:
$$
\begin{align*}
\mathcal{P}_{\star}^{\text{AM}} = \max_{\tau\leq T} \mathbb{E}_{\mathbb{Q}}\Bigl(\mathcal{D}^{-1}_{\tau,0}(g_{y})\cdot V_{\star}(K,S(\tau))\Bigr) \geq \mathcal{P}_{\star}^{\text{EU}}\qquad\text{for }\star\in\{c,p\}
\end{align*}
$$
where $\mathcal{D}^{-1}_{\tau,0}(g_{y})$ discounts the payoff $V_{\star}(K,S(\tau))$ of a contract with strike price $K$ and expiration $T$ from the exercise time $\tau$ back to today at the risk-free rate $g_{y}$, and $\tau$ is chosen using only the information available at that time (a __stopping time__). 

The American premium is at least the European one because the American holder can always wait until expiration. The standard method to price American options is the Cox, Ross, Rubinstein (CRR) binomial model.

__Reference:__
* [Cox, J. C.; Ross, S. A.; Rubinstein, M. (1979). "Option pricing: A simplified approach". Journal of Financial Economics. 7 (3): 229. CiteSeerX 10.1.1.379.7582. doi:10.1016/0304-405X(79)90015-1](https://www.sciencedirect.com/science/article/pii/0304405X79900151)

### CRR Binomial Lattice Model
The CRR model assumes that the movement of the share price of the underlying asset can be modeled as a binomial tree, where at each time step, the price can either move up by a factor of $u$ or down by a factor of $d$. The probability of an up move is the __risk-neutral probability__ $q$, and the probability of a down move is $(1-q)$.

> __Risk-Neutral, what?__ In the CRR approach, the binomial tree parameters are no longer data driven. Models compute the $u$, $d$, and $q$ parameters. The risk-neutral probability $q$ is derived such that the expected return of the underlying asset matches the risk-free rate $g_{y}$.

Let $S_{0}$ denote the current share price (units: USD/share). One time step in the future, the expected share price under the risk-neutral pricing rule $\mathbb{Q}$ is given by:
$$
\begin{align*}
\mathcal{D}_{1,0}(g_{y})\cdot S_{0} & = \mathbb{E}_{\mathbb{Q}}\left(S_{1}\right) && \text{grow at the risk-free rate under }\mathbb{Q}\\
& = q\cdot(u\cdot S_{0}) + (1-q)\cdot(d\cdot S_{0}) && \text{up with probability } q\text{, down with } (1-q)\\
& = S_{0}\bigl(qu + (1-q)d\bigr) && \text{factor out } S_{0}\\
& = S_{0}\bigl(qu + d - qd\bigr) && \text{expand } (1-q)d\\
& = S_{0}\bigl(d + q(u - d)\bigr) && \text{collect the } q \text{ terms}
\end{align*}
$$
where $\mathcal{D}_{1,0}(g_{y})=e^{g_{y}\Delta t}$ is the risk-free accumulation factor for one time step of length $\Delta t$ (units: years). Dividing by the initial share price $S_{0}$ and solving for $q$ gives:
$$
\boxed{
\begin{align*}
q = \frac{\mathcal{D}_{1,0}\left(g_{y}\right) - d}{u - d}\quad\blacksquare
\end{align*}}
$$
This is the risk-neutral probability we found by ruling out arbitrage when we introduced lattice models, with $R_{f}=\mathcal{D}_{1,0}(g_{y})$. However, unlike the data-driven binomial model we discussed earlier, the CRR model derives $u$ and $d$ based on the volatility $\sigma$ of the underlying asset, the time step $\Delta t$, and the symmetric condition $ud = 1$:
$$
\boxed{
\begin{align*}
u &= e^{\sigma \sqrt{\Delta t}} \\
d &= e^{-\sigma \sqrt{\Delta t}}
\end{align*}}
$$
where $\sigma$ is the volatility (units: $\mathrm{yr}^{-1/2}$). [Want more on where this comes from?](advanced/crr_factors/CHEME-5660-L9b-Advanced-CRR-Factors-Fall-2026.ipynb) Let's use this lattice to price American contracts.

### Decision-Rule for American Options
The ability to exercise early introduces an additional decision-making layer in the pricing process beyond that of European options. 

>__Decision Rule__: At each node in the binomial tree, the holder of an American option must decide whether to exercise the option immediately or to hold onto it for potential future gains. The decision is based on comparing the immediate exercise value with the expected value of holding the option discounted back to the present value.

Let's look at a schematic of the decision rule for an American option contract.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L9b-American-Decision-Rule.svg" width="700" alt="Left, a three-step binomial lattice from t = 0 (now) to t = 3 that starts at the share price S0. The first step is labeled (u, q) for an up move and (d, 1 − q) for a down move, and the prices at t = 3 are u³S0, u²dS0, ud²S0, and d³S0. The top cell from t = 2 to t = 3 is shaded and enlarged in a panel on the right: the node value V̂(2,2) branches to V̂(3,3) with (u, q) and to V̂(3,2) with (d, 1 − q). Below the panel is the decision rule for an American contract: the node value is the larger of the payoff V(K, S) from exercising now and the discounted risk-neutral expectation of the two next values from holding for the future."/>
    </center>
</div>

At expiration, each node's value is the contract's payoff. Working backward, the American holder takes the larger of exercising now and holding for the future, while the European holder can only hold:
$$
\boxed{
\begin{align*}
\hat{V}^{\text{AM}}_{t,k} & = \max\Bigl\{\underbrace{V(K,S_{t,k})}_{\text{exercise now}},~\underbrace{\mathcal{D}^{-1}_{t+1,t}(g_{y})\cdot\overbrace{\bigl(q\,\hat{V}^{\text{AM}}_{t+1,k+1} + (1-q)\,\hat{V}^{\text{AM}}_{t+1,k}\bigr)}^{\mathbb{E}_{\mathbb{Q}}}}_{\text{hold for the future}}\Bigr\}\\
\hat{V}^{\text{EU}}_{t,k} & = \mathcal{D}^{-1}_{t+1,t}(g_{y})\cdot\overbrace{\bigl(q\,\hat{V}^{\text{EU}}_{t+1,k+1} + (1-q)\,\hat{V}^{\text{EU}}_{t+1,k}\bigr)}^{\mathbb{E}_{\mathbb{Q}}}
\end{align*}}
$$
where $\hat{V}_{t,k}$ is the contract's value at the node with $k$ up moves in $t$ steps, $S_{t,k}=S_{0}u^{k}d^{t-k}$ is the share price there, and $\mathcal{D}^{-1}_{t+1,t}(g_{y})=e^{-g_{y}\Delta t}$ discounts one time step. The premium is the value at the root, $\mathcal{P}=\hat{V}_{0,0}$. The difference $\mathcal{P}^{\text{AM}}-\mathcal{P}^{\text{EU}}\geq 0$ is the value of the right to exercise early, the __early-exercise premium__.

Let's outline some pseudocode for pricing an option contract on the CRR lattice.

__Initialize__: Given the share price $S_{0}$, the strike price $K$, the payoff function $V(K,S)$ of the call or put, the time to expiration $T$ (units: years), the volatility $\sigma$, the risk-free rate $g_{y}$, and the number of time steps $h$. Compute the time step $\Delta t = T/h$, the up and down factors $u$ and $d$, the risk-neutral probability $q$, and the one-step discount factor $\mathcal{D}^{-1}_{t+1,t}(g_{y})=e^{-g_{y}\Delta t}$.

1. Build the share price lattice: $S_{t,k} \gets S_{0}u^{k}d^{t-k}$ for each level $t = 0,1,\dots,h$ and node $k = 0,1,\dots,t$.
2. At expiration, set each node value to the payoff: $\hat{V}_{h,k} \gets V(K,S_{h,k})$ for $k = 0,1,\dots,h$.
3. For each level $t = h-1, h-2, \dots, 0$ __do__:
   - For each node $k = 0,1,\dots,t$ __do__:
      - a. Compute the value of holding for the future: $H_{t,k} \gets \mathcal{D}^{-1}_{t+1,t}(g_{y})\cdot\bigl(q\,\hat{V}_{t+1,k+1} + (1-q)\,\hat{V}_{t+1,k}\bigr)$.
      - b. If the contract is American, __then__: $\hat{V}_{t,k} \gets \max\bigl\{V(K,S_{t,k}),~H_{t,k}\bigr\}$. Otherwise, $\hat{V}_{t,k} \gets H_{t,k}$.
4. The premium is the value at the root: $\mathcal{P} \gets \hat{V}_{0,0}$.

Before we dig into the implementation on real data, let's look at a simple hypothetical example from Chapter 13 of Hull that compares an American and European option contract price.

> __Example:__ Consider a put contract. The contract's strike price is $K$ = 52 USD/share on a stock whose current price is $S_{0}$ = 50 USD/share. There are two time steps of 1 year each. In each period, the stock price moves up by a factor of $u$ = 1.2 or down by a factor of $d$ = 0.8. Let the continuously compounded risk-free rate be $g_{y}$ = 5%, so the risk-neutral probability of an up move is $q=(e^{0.05}-0.8)/(1.2-0.8)$ = 0.6282.

What is the price of the American and European put option contracts?

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-HullExample-American-v-European-Schematic.svg" width="640" alt="Hull's two-step put tree with K = 52, S0 = 50, u = 1.2, and d = 0.8, American contract on the left and European on the right. At the down node at t = 1 (share price 40), the American holder exercises for 12 instead of holding for 9.46, so the premiums at the root are 5.09 and 4.19 USD/share."/>
    </center>
</div>

At the down node at $t=1$, exercising pays 12 USD/share, while holding is worth 9.46 USD/share, so the American holder exercises. That one decision raises the premium from 4.19 to 5.09 USD/share, an early-exercise premium of about 0.90 USD/share.

Let's compute every node of this lattice, then apply the same method to contracts from the course's options archive:

> __Examples:__
>
> __Review on your own:__ [▶ Let's compute the Premium for Hull's Hypothetical American Put using a binomial lattice](CHEME-5660-L9b-Example-Hull-American-Put-Fall-2026.ipynb). In this example, we'll compute the price of a hypothetical American put option contract using backward induction over a binomial lattice model, and compare our results to the same contract written as a European contract.
>
> __We'll work through in lecture:__ [▶ Let's compute the Premium for Market-Quoted American Call and Put Options using CRR](CHEME-5660-L9b-Example-AmericanContractsPremium-Fall-2026.ipynb). In this example, we'll compute the price of American call and put option contracts using the CRR binomial model, compare our results to observed market prices, and measure each contract's early-exercise premium.

Having priced Hull's put and the archived contracts on the CRR lattice, let's look at a second way to compute the American premium.

### Monte Carlo Pricing of American Options
Previously, we priced European contracts by simulating share prices under the risk-neutral measure $\mathbb{Q}$ and averaging the discounted payoffs. Longstaff and Schwartz extended this approach to American contracts.

__Reference:__
* [Longstaff, F. A.; Schwartz, E. S. (2001). "Valuing American Options by Simulation: A Simple Least-Squares Approach". The Review of Financial Studies. 14 (1): 113–147. doi:10.1093/rfs/14.1.113](https://doi.org/10.1093/rfs/14.1.113)

We simulate many share price paths under $\mathbb{Q}$, then estimate the expected value of holding at each date by regressing the discounted cash flows from holding on the current share price across paths. Like the lattice, Longstaff and Schwartz work backward from expiration:

__Initialize__: Given the share price $S_{0}$, the strike price $K$, the payoff function $V(K,S)$ of the call or put, the time to expiration $T$ (units: years), the volatility $\sigma$, the risk-free rate $g_{y}$, the number of exercise dates $N$ after today, and the number of paths $M$. Compute the time between exercise dates $\Delta t = T/N$ and the one-step discount factor $\mathcal{D}^{-1}_{t+1,t}(g_{y})=e^{-g_{y}\Delta t}$.

1. __Simulate:__ Generate $M$ share price paths under $\mathbb{Q}$ on the exercise dates $t = 0,1,\dots,N$, where $S_{i}(t)$ is the share price on path $i$ at date $t$.
2. At expiration, set each path's cash flow to its payoff: $Y_{i} \gets V(K,S_{i}(N))$ for $i = 1,2,\dots,M$.
3. For each date $t = N-1, N-2, \dots, 1$ __do__:
   - a. Discount each cash flow back one date: $Y_{i} \gets \mathcal{D}^{-1}_{t+1,t}(g_{y})\cdot Y_{i}$.
   - b. __Regress:__ On the in-the-money paths, those with $V(K,S_{i}(t)) > 0$, fit $\hat{C}(S)=\beta_{0}+\beta_{1}S+\beta_{2}S^{2}$ to the pairs $\bigl(S_{i}(t),Y_{i}\bigr)$ by least squares. The fitted curve estimates the value of holding.
   - c. __Decide:__ For each in-the-money path, if $V(K,S_{i}(t)) > \hat{C}(S_{i}(t))$, __then__: exercise, replacing the later cash flow with the payoff now, $Y_{i} \gets V(K,S_{i}(t))$.
4. Discount the cash flows to today and average them: $\bar{Y} \gets \mathcal{D}^{-1}_{1,0}(g_{y})\cdot\frac{1}{M}\sum_{i=1}^{M}Y_{i}$.
5. The premium is the larger of exercising today and holding: $\mathcal{P}^{\text{AM}} \gets \max\bigl\{V(K,S_{0}),~\bar{Y}\bigr\}$.

Let's price an American put by simulation and compare it with the lattice and the European premium:

> __Example:__
>
> [▶ Let's compute the premium of an American put by Monte Carlo](CHEME-5660-L9b-Example-LSM-Premium-Fall-2026.ipynb). In this example, we'll work the Longstaff–Schwartz regression by hand on eight paths, then price an American put by simulation and compare it with the CRR lattice and the European premium.

Holding on every path gives the European Monte Carlo premium, so the difference estimates the early-exercise premium.
___

## Optional Advanced Material

The [advanced index](advanced/README.md) describes the optional notebook below:

* [▶ Deriving the CRR up and down factors](advanced/crr_factors/CHEME-5660-L9b-Advanced-CRR-Factors-Fall-2026.ipynb). In this notebook, we derive the up and down factors of the CRR model by matching the variance of the lattice's one-step log return to the variance of geometric Brownian motion. We then check that the match holds under the risk-neutral probability, and find how small the time step must be to keep that probability between zero and one. It is a derivation notebook with no code.

___

## Summary
This lecture explored how American options are priced using the CRR binomial model and Monte Carlo simulation.

> __Key Takeaways:__
>
> * __Binomial trees price American options:__ The CRR model creates a discrete price lattice where at each node, the holder decides whether to exercise immediately or hold the option, making backward induction necessary to compute prices.
> * __The decision rule sets the early-exercise premium:__ We priced Hull's two-step put with the American and European rules on the same lattice. The American holder exercises at the one node where exercising pays more than holding, and that decision accounts for the whole difference between the two premiums.
> * __Simulation estimates the same premium:__ Longstaff–Schwartz works backward along simulated paths, regressing the value of holding on the share price. Holding on every path gives the European premium.

The early-exercise premium is what the right to exercise before expiration is worth, and the lattice and simulation give two ways to compute it.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___